# Zero-shot Evaluation - Original Prompt

Avalia os seis modelos sem fine-tuning usando exatamente o prompt/instruction do JSONL processado. Os resultados são salvos em `data/processed/zero-shot/<modelo>/`.

## 1. Setup

In [1]:
from __future__ import annotations

import csv
import json
import math
import os
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any


def find_project_root(marker: str = ".git") -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"

In [2]:
def configure_mlx_cuda13_env() -> None:
    """Expose CUDA shared libraries installed inside the uv-managed .venv."""
    import site

    site_packages_candidates = [Path(p) for p in site.getsitepackages()]
    site_packages_candidates.append(PROJECT_ROOT / ".venv" / "lib" / "python3.13" / "site-packages")

    cuda_dirs: list[Path] = []
    for site_packages in site_packages_candidates:
        for relative in ["nvidia/cu13/lib", "nvidia/cudnn/lib", "nvidia/nccl/lib"]:
            path = site_packages / relative
            if path.exists() and path not in cuda_dirs:
                cuda_dirs.append(path)

    current = os.environ.get("LD_LIBRARY_PATH", "")
    cuda_path = ":".join(str(path) for path in cuda_dirs)
    os.environ["LD_LIBRARY_PATH"] = f"{cuda_path}:{current}" if current else cuda_path
    os.environ.setdefault("MLX_CUDA_SDPA_CACHE_SIZE", "512")


configure_mlx_cuda13_env()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("LD_LIBRARY_PATH configured for MLX CUDA:", bool(os.environ.get("LD_LIBRARY_PATH")))
print("MLX_CUDA_SDPA_CACHE_SIZE:", os.environ.get("MLX_CUDA_SDPA_CACHE_SIZE"))

PROJECT_ROOT: /Documents/slm-nes/aibox-nes-slms
LD_LIBRARY_PATH configured for MLX CUDA: True
MLX_CUDA_SDPA_CACHE_SIZE: 512


## 2. Modelos e Funções

In [3]:
COMPETENCIES = [
    "formal_register",
    "thematic_coherence",
    "narrative_rhetorical_structure",
    "cohesion",
]

STRICT_PROMPT_TEMPLATE = """Você é um avaliador de redações narrativas em português brasileiro do ensino fundamental.

Avalie a redação abaixo em quatro competências:
1. formal_register
2. thematic_coherence
3. narrative_rhetorical_structure
4. cohesion

Cada competência deve receber uma nota inteira de 1 a 5.

Responda exclusivamente com um objeto JSON válido.
Não inclua explicações.
Não use markdown.
Não escreva texto antes ou depois do JSON.
Use exatamente estas quatro chaves:
"formal_register", "thematic_coherence", "narrative_rhetorical_structure", "cohesion".

Formato obrigatório:
{{"formal_register": 1, "thematic_coherence": 1, "narrative_rhetorical_structure": 1, "cohesion": 1}}

Tema:
{prompt}

Redação:
{essay}
"""


@dataclass(frozen=True)
class ModelSpec:
    slug: str
    model_id: str
    max_tokens: int = 48


MODEL_SPECS = [
    ModelSpec("smol-lm-135m-4bit", "mlx-community/SmolLM-135M-4bit"),
    ModelSpec("smol-lm-135m-it-4bit", "mlx-community/SmolLM-135M-Instruct-4bit"),
    ModelSpec("smol-lm-360m-it", "mlx-community/SmolLM-360M-Instruct"),
    ModelSpec("gemma-3-270m-it-4bit", "mlx-community/gemma-3-270m-it-4bit"),
    ModelSpec("gemma-3-1b-it-4bit", "mlx-community/gemma-3-1b-it-4bit"),
    ModelSpec("qwen2.5-0.5b-it-4bit", "mlx-community/Qwen2.5-0.5B-Instruct-4bit"),
]

MODEL_SPECS

[ModelSpec(slug='smol-lm-135m-4bit', model_id='mlx-community/SmolLM-135M-4bit', max_tokens=48),
 ModelSpec(slug='smol-lm-135m-it-4bit', model_id='mlx-community/SmolLM-135M-Instruct-4bit', max_tokens=48),
 ModelSpec(slug='smol-lm-360m-it', model_id='mlx-community/SmolLM-360M-Instruct', max_tokens=48),
 ModelSpec(slug='gemma-3-270m-it-4bit', model_id='mlx-community/gemma-3-270m-it-4bit', max_tokens=48),
 ModelSpec(slug='gemma-3-1b-it-4bit', model_id='mlx-community/gemma-3-1b-it-4bit', max_tokens=48),
 ModelSpec(slug='qwen2.5-0.5b-it-4bit', model_id='mlx-community/Qwen2.5-0.5B-Instruct-4bit', max_tokens=48)]

In [4]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open(encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_number}") from exc
    return rows


def load_raw_csv(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open(encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            output = {
                "formal_register": int(row["formal_register"]),
                "thematic_coherence": int(row["thematic_coherence"]),
                "narrative_rhetorical_structure": int(row["narrative_rhetorical_structure"]),
                "cohesion": int(row["cohesion"]),
            }
            rows.append(
                {
                    "id": row.get("id", ""),
                    "prompt": row["prompt"],
                    "essay": row["essay"],
                    "instruction": STRICT_PROMPT_TEMPLATE.format(prompt=row["prompt"], essay=row["essay"]),
                    "output": json.dumps(output, ensure_ascii=False),
                    "dataset_source": path.name,
                }
            )
    return rows


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_json(path: Path, data: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

In [5]:
def parse_evaluation_output(output: str) -> dict[str, int] | None:
    if not isinstance(output, str) or output.startswith("ERROR"):
        return None

    candidates = [output.strip()]
    match = re.search(r"\{.*?\}", output, flags=re.DOTALL)
    if match:
        candidates.insert(0, match.group(0))

    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            continue

        if not isinstance(parsed, dict):
            continue

        normalized: dict[str, int] = {}
        for key in COMPETENCIES:
            if key not in parsed:
                break
            try:
                value = int(parsed[key])
            except (TypeError, ValueError):
                break
            if value < 1 or value > 5:
                break
            normalized[key] = value

        if len(normalized) == len(COMPETENCIES):
            return normalized

    return None


def weighted_classification_metrics(y_true: list[int], y_pred: list[int]) -> dict[str, float]:
    labels = sorted(set(y_true) | set(y_pred))
    total = len(y_true)
    correct = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
    accuracy = correct / total if total else math.nan

    weighted_precision = 0.0
    weighted_recall = 0.0
    weighted_f1 = 0.0

    for label in labels:
        support = sum(1 for value in y_true if value == label)
        if support == 0:
            continue

        true_positive = sum(1 for true, pred in zip(y_true, y_pred) if true == label and pred == label)
        predicted_positive = sum(1 for pred in y_pred if pred == label)
        precision = true_positive / predicted_positive if predicted_positive else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0.0

        weight = support / total
        weighted_precision += weight * precision
        weighted_recall += weight * recall
        weighted_f1 += weight * f1

    return {
        "accuracy": accuracy,
        "precision": weighted_precision,
        "recall": weighted_recall,
        "f1_score": weighted_f1,
    }


def confusion_matrix_1_to_5(y_true: list[int], y_pred: list[int]) -> list[list[int]]:
    return [
        [sum(1 for true, pred in zip(y_true, y_pred) if true == i and pred == j) for j in range(1, 6)]
        for i in range(1, 6)
    ]


def evaluate_results(rows: list[dict[str, Any]]) -> dict[str, Any]:
    expected_parsed: list[dict[str, int]] = []
    generated_parsed: list[dict[str, int]] = []
    parsing_errors = 0
    total_samples = len(rows)

    for row in rows:
        expected = parse_evaluation_output(row.get("expected_output", ""))
        generated = parse_evaluation_output(row.get("generated_output", ""))
        if expected is None or generated is None:
            parsing_errors += 1
            continue
        expected_parsed.append(expected)
        generated_parsed.append(generated)

    results: dict[str, Any] = {
        "total_samples": total_samples,
        "valid_samples": len(expected_parsed),
        "parsing_errors": parsing_errors,
        "valid_output_rate": len(expected_parsed) / total_samples if total_samples else math.nan,
        "competencies": {},
        "overall": {},
        "end_to_end": {},
    }

    if not expected_parsed:
        results["end_to_end"] = {
            "accuracy": 0.0 if total_samples else math.nan,
            "precision": 0.0 if total_samples else math.nan,
            "recall": 0.0 if total_samples else math.nan,
            "f1_score": 0.0 if total_samples else math.nan,
            "exact_match_rate": 0.0 if total_samples else math.nan,
            "off_by_one_rate": 0.0 if total_samples else math.nan,
        }
        return results

    all_y_true: list[int] = []
    all_y_pred: list[int] = []
    validity_weight = len(expected_parsed) / total_samples if total_samples else math.nan

    for competency in COMPETENCIES:
        y_true = [sample[competency] for sample in expected_parsed]
        y_pred = [sample[competency] for sample in generated_parsed]
        metrics = weighted_classification_metrics(y_true, y_pred)
        exact_matches = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
        off_by_one = sum(1 for true, pred in zip(y_true, y_pred) if abs(true - pred) <= 1)

        results["competencies"][competency] = {
            **metrics,
            "exact_match_rate": exact_matches / len(y_true),
            "off_by_one_rate": off_by_one / len(y_true),
            "end_to_end_accuracy": metrics["accuracy"] * validity_weight,
            "end_to_end_precision": metrics["precision"] * validity_weight,
            "end_to_end_recall": metrics["recall"] * validity_weight,
            "end_to_end_f1_score": metrics["f1_score"] * validity_weight,
            "end_to_end_exact_match_rate": exact_matches / total_samples,
            "end_to_end_off_by_one_rate": off_by_one / total_samples,
            "confusion_matrix": confusion_matrix_1_to_5(y_true, y_pred),
        }

        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)

    overall_metrics = weighted_classification_metrics(all_y_true, all_y_pred)
    overall_exact = sum(1 for true, pred in zip(all_y_true, all_y_pred) if true == pred)
    overall_off_by_one = sum(1 for true, pred in zip(all_y_true, all_y_pred) if abs(true - pred) <= 1)

    results["overall"] = {
        **overall_metrics,
        "exact_match_rate": overall_exact / len(all_y_true),
        "off_by_one_rate": overall_off_by_one / len(all_y_true),
    }
    results["end_to_end"] = {
        "accuracy": overall_metrics["accuracy"] * validity_weight,
        "precision": overall_metrics["precision"] * validity_weight,
        "recall": overall_metrics["recall"] * validity_weight,
        "f1_score": overall_metrics["f1_score"] * validity_weight,
        "exact_match_rate": overall_exact / (total_samples * len(COMPETENCIES)),
        "off_by_one_rate": overall_off_by_one / (total_samples * len(COMPETENCIES)),
    }
    return results

In [6]:
def apply_chat_template(tokenizer: Any, instruction: str) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": instruction}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        if isinstance(prompt, str):
            return prompt
    return instruction


def extract_json_response(response: str) -> str:
    response = response.strip()
    match = re.search(r"\{.*?\}", response, flags=re.DOTALL)
    return match.group(0).strip() if match else response


def load_existing_results(path: Path) -> dict[int, dict[str, Any]]:
    if not path.exists():
        return {}
    rows = load_jsonl(path)
    return {int(row["sample_id"]): row for row in rows if "sample_id" in row and isinstance(row.get("sample_id"), int)}


def run_model(
    spec: ModelSpec,
    test_rows: list[dict[str, Any]],
    output_dir: Path,
    *,
    limit: int | None = None,
    resume: bool = True,
    overwrite: bool = False,
    verbose: bool = False,
    prompt_mode: str,
) -> dict[str, Any]:
    from mlx_lm.generate import generate
    from mlx_lm.utils import load

    output_dir.mkdir(parents=True, exist_ok=True)
    results_path = output_dir / "test_results.jsonl"
    metrics_path = output_dir / "evaluation_metrics.json"
    metadata_path = output_dir / "run_metadata.json"

    if overwrite and results_path.exists():
        results_path.unlink()

    existing = load_existing_results(results_path) if resume else {}
    selected_rows = test_rows[:limit] if limit is not None else test_rows

    print(f"\n== {spec.slug} ==")
    print(f"Loading {spec.model_id}")
    model, tokenizer, *_ = load(spec.model_id)

    started_at = time.time()
    for sample_id, sample in enumerate(selected_rows):
        if sample_id in existing:
            continue

        instruction = str(sample["instruction"])
        expected_output = str(sample.get("output", ""))
        prompt = apply_chat_template(tokenizer, instruction)

        try:
            generated = generate(model, tokenizer, prompt=prompt, max_tokens=spec.max_tokens, verbose=verbose)
            generated_output = extract_json_response(generated)
        except Exception as exc:
            generated_output = f"ERROR: {type(exc).__name__}: {exc}"

        append_jsonl(
            results_path,
            {
                "sample_id": sample_id,
                "instruction": instruction,
                "expected_output": expected_output,
                "generated_output": generated_output,
                "dataset_source": sample.get("dataset_source", "test.jsonl"),
            },
        )

        if (sample_id + 1) % 10 == 0 or sample_id + 1 == len(selected_rows):
            print(f"Processed {sample_id + 1}/{len(selected_rows)}")

    rows = load_jsonl(results_path)
    metrics = evaluate_results(rows)
    write_json(metrics_path, metrics)
    write_json(
        metadata_path,
        {
            "mode": "zero-shot",
            "prompt_mode": prompt_mode,
            "slug": spec.slug,
            "model_id": spec.model_id,
            "test_samples_requested": len(selected_rows),
            "results_file": str(results_path),
            "metrics_file": str(metrics_path),
            "elapsed_seconds": round(time.time() - started_at, 3),
        },
    )
    print(
        "Saved metrics: "
        f"valid={metrics['valid_samples']}/{metrics['total_samples']} "
        f"overall_f1={metrics.get('overall', {}).get('f1_score')}"
    )
    return metrics


def run_all_models(test_rows: list[dict[str, Any]], output_root: Path, *, prompt_mode: str, limit: int | None = None, resume: bool = True, overwrite: bool = False) -> None:
    for spec in MODEL_SPECS:
        run_model(
            spec,
            test_rows,
            output_root / spec.slug,
            limit=limit,
            resume=resume,
            overwrite=overwrite,
            prompt_mode=prompt_mode,
        )

In [7]:
def summarize_output_root(output_root: Path) -> list[dict[str, Any]]:
    rows = []
    for metrics_path in sorted(output_root.glob("*/evaluation_metrics.json")):
        data = json.loads(metrics_path.read_text(encoding="utf-8"))
        end_to_end = data.get("end_to_end", {})
        rows.append(
            {
                "model": metrics_path.parent.name,
                "valid_samples": data.get("valid_samples"),
                "total_samples": data.get("total_samples"),
                "valid_output_rate": data.get("valid_output_rate"),
                "overall_f1_valid_only": data.get("overall", {}).get("f1_score"),
                "end_to_end_f1": end_to_end.get("f1_score"),
                "end_to_end_off_by_one": end_to_end.get("off_by_one_rate"),
            }
        )
    return rows


def show_examples(output_root: Path, model_slug: str, n: int = 3) -> None:
    path = output_root / model_slug / "test_results.jsonl"
    for idx, line in enumerate(path.open(encoding="utf-8")):
        row = json.loads(line)
        print(f"sample {row['sample_id']}: {row['generated_output'].replace(chr(10), ' ')[:500]}")
        if idx + 1 >= n:
            break

## 3. Carregar Teste

In [8]:
PROMPT_MODE = "original"
TEST_FILE = PROCESSED_DIR / "test.jsonl"
OUTPUT_ROOT = PROCESSED_DIR / "zero-shot"
test_rows = load_jsonl(TEST_FILE)
len(test_rows), OUTPUT_ROOT

(370, PosixPath('/Documents/slm-nes/aibox-nes-slms/data/processed/zero-shot'))

## 4. Smoke Test Opcional

In [ ]:
# Descomente para validar uma amostra.
# run_model(MODEL_SPECS[1], test_rows, OUTPUT_ROOT / MODEL_SPECS[1].slug, limit=1, overwrite=True, resume=False, prompt_mode=PROMPT_MODE)

## 5. Rodar Todos os Modelos

In [ ]:
# Mantém --resume por padrão para continuar execuções interrompidas.
# Descomente para rerodar/coletar.
# run_all_models(test_rows, OUTPUT_ROOT, prompt_mode=PROMPT_MODE, resume=True, overwrite=False)

## 6. Resumo dos Resultados Coletados

In [9]:
import pandas as pd
summary = pd.DataFrame(summarize_output_root(OUTPUT_ROOT))
summary

,model,valid_samples,total_samples,valid_output_rate,overall_f1_valid_only,end_to_end_f1,end_to_end_off_by_one
0,gemma-3-1b-it-4bit,0,370,0.0,None,0.0,0.0
1,gemma-3-270m-it-4bit,0,370,0.0,None,0.0,0.0
2,qwen2.5-0.5b-it-4bit,0,370,0.0,None,0.0,0.0
3,smol-lm-135m-4bit,0,370,0.0,None,0.0,0.0
4,smol-lm-135m-it-4bit,0,370,0.0,None,0.0,0.0
5,smol-lm-360m-it,0,370,0.0,None,0.0,0.0


## 7. Exemplos de Saída

In [10]:
# show_examples(OUTPUT_ROOT, "qwen2.5-0.5b-it-4bit", n=3)